# KumoRFM Tutorial: Forecast 21-day SKU Demand from Relational Data

**Forecast-only**: This tutorial demonstrates how to use KumoRFM to predict 21-day demand per SKU from relational (multi-table) retail data. **It produces demand forecasts, not ordering decisions.**

## What this tutorial covers

- Building a temporal, relational graph (sales events + skus + products + stores)
- Handling availability/censorship by omitting out-of-stock weeks from events
- Preventing time leakage via explicit as-of date truncation
- Using Predictive Query Language (PQL) to forecast total demand over the next 21 days
- Saving predictions alongside minimal run metadata for reproducibility
- Optional mini backtest with sliding as-of dates (MAE/MAPE validation)

## What this tutorial does NOT cover

- Inventory policies, cost optimization, or safety stock calculations
- Order quantity computation or submission generation
- Hyperparameter tuning or cross-validation

**Forecasts ≠ Orders**. The predictions from this notebook can be inputs to downstream decision models.

## Attribution

- **KumoRFM**: [kumo.ai](https://kumo.ai) | [GitHub](https://github.com/kumo-ai/kumo-rfm)
- **Dataset**: VN2 Inventory Optimization Challenge (retail sales, product/store hierarchy)
- **Tutorial author**: Demonstration of KumoRFM's multi-hop temporal forecasting on real retail data


## Setup


In [ ]:
# Install KumoRFM SDK (colab/notebook-friendly)
%pip install -q kumoai --pre --upgrade

import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import kumoai.experimental.rfm as rfm

# Project paths
PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
OUT_DIR = PROJECT_ROOT / "artifacts/kumo_rfm"  # Outputs (predictions CSV + metadata)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Authenticate via environment variable (NO secrets in code!)
KUMO_API_KEY = os.environ.get("KUMO_API_KEY")
if not KUMO_API_KEY:
    # If running interactively, rfm.authenticate() opens a widget
    rfm.authenticate()

rfm.init()
print("✅ KumoRFM initialized")



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


[2025-10-09 17:27:16 - kumoai:203 - INFO] Successfully initialized the Kumo SDK against deployment https://kumorfm.ai/api, with log level INFO.


✅ KumoRFM initialized


## Load data and apply availability-aware masking (censorship)


In [2]:
# Inputs (adjust filenames as needed)
INDEX = ["Store", "Product"]
AS_OF_DATE = pd.Timestamp("2024-04-08")  # Week 0: cutoff to prevent time leakage

# Load wide tables
sales_wide = pd.read_csv(DATA_DIR / "Week 0 - 2024-04-08 - Sales.csv")
avail_wide = pd.read_csv(DATA_DIR / "Week 0 - In Stock.csv")
master = pd.read_csv(DATA_DIR / "Week 0 - Master.csv")

# Availability-aware mask: omit OOS weeks from events
sw = sales_wide.set_index(INDEX).copy()
aw = avail_wide.set_index(INDEX).copy()
sw.columns = pd.to_datetime(sw.columns)
aw.columns = pd.to_datetime(aw.columns)

avail = aw.astype(bool)
sales_c = sw.where(avail)  # OOS -> NaN, true zeros preserved

# Melt to long events
sales_long = (
    sales_c.reset_index()
    .melt(id_vars=INDEX, var_name="week", value_name="qty")
    .rename(columns={"Store": "store_id", "Product": "product_id"})
)

sales_long = sales_long.dropna(subset=["qty"]).astype({"qty": float})
sales_long["week"] = pd.to_datetime(sales_long["week"]) 

# CRITICAL: Truncate to as-of date to prevent time leakage
# RFM will predict forward from this anchor; only past events are used as context
sales_long = sales_long[sales_long["week"] <= AS_OF_DATE]

print({
    "events": len(sales_long),
    "skus": sales_long[["store_id","product_id"]].drop_duplicates().shape[0],
    "date_min": str(sales_long["week"].min()),
    "date_max": str(sales_long["week"].max()),
    "as_of": str(AS_OF_DATE.date()),
})


{'events': 83526, 'skus': 599, 'date_min': '2021-04-12 00:00:00', 'date_max': '2024-04-08 00:00:00', 'as_of': '2024-04-08'}


## Build relational tables: sales, skus, products, stores


In [3]:
# SKUs: one row per (store_id, product_id)
skus = sales_wide[INDEX].drop_duplicates().rename(columns={"Store":"store_id","Product":"product_id"})
skus["sku_id"] = range(len(skus))

# PRODUCTS: product hierarchy (use what's available)
prod_cols = ["Product", "Department"] + [c for c in ["Division","ProductGroup","Brand","Category"] if c in master.columns]
products = master[prod_cols].drop_duplicates().rename(columns={"Product":"product_id"})
for c in products.columns:
    if c != "product_id" and products[c].dtype == object:
        products[c] = products[c].fillna("Unknown")

# STORES: store attributes if present
store_cols = ["Store"] + [c for c in ["Region","Cluster","StoreSize","StoreType"] if c in master.columns]
stores = master[store_cols].drop_duplicates().rename(columns={"Store":"store_id"})
for c in stores.columns:
    if c != "store_id" and stores[c].dtype == object:
        stores[c] = stores[c].fillna("Unknown")

# SALES events: add sku_id FK
sales = sales_long.merge(skus, on=["store_id","product_id"], how="left")[
    ["sku_id","store_id","product_id","week","qty"]
].copy()

print({
    "sales_rows": len(sales),
    "skus": len(skus),
    "products": len(products),
    "stores": len(stores)
})


{'sales_rows': 83526, 'skus': 599, 'products': 297, 'stores': 67}


## Create KumoRFM graph and declare schema (stypes, time column, links)


In [4]:
# Build LocalGraph from dataframes (let RFM infer, then adjust)
graph = rfm.LocalGraph.from_data({
    'sales': sales,
    'skus': skus,
    'products': products,
    'stores': stores,
}, infer_metadata=True)

# Explicit schema: time and stypes
graph['sales'].time_column = 'week'

graph['sales']['qty'].stype = 'numerical'
graph['sales']['sku_id'].stype = 'ID'
graph['sales']['store_id'].stype = 'ID'
graph['sales']['product_id'].stype = 'ID'

graph['skus']['sku_id'].stype = 'ID'
graph['skus']['store_id'].stype = 'ID'
graph['skus']['product_id'].stype = 'ID'

graph['products']['product_id'].stype = 'ID'
for c in products.columns:
    if c != 'product_id':
        graph['products'][c].stype = 'categorical'

graph['stores']['store_id'].stype = 'ID'
for c in stores.columns:
    if c != 'store_id':
        graph['stores'][c].stype = 'categorical'

# Explicit links (FK→PK)
try:
    graph.link(src_table='sales', fkey='sku_id', dst_table='skus')
    graph.link(src_table='skus', fkey='product_id', dst_table='products')
    graph.link(src_table='skus', fkey='store_id', dst_table='stores')
except Exception as e:
    print(f"Link note: {e}")

print("✅ Graph ready (time column set, stypes assigned, links declared)")


### 🗂️ Graph Metadata

name,primary_key,time_column
sales,-,week
skus,sku_id,-
products,product_id,-
stores,store_id,-


### 🕸️ Graph Links (FK ↔️ PK)

- `sales.product_id` ↔️ `products.product_id`
- `skus.product_id` ↔️ `products.product_id`
- `sales.sku_id` ↔️ `skus.sku_id`
- `sales.store_id` ↔️ `stores.store_id`
- `skus.store_id` ↔️ `stores.store_id`

Link note: Edge(src_table='sales', fkey='sku_id', dst_table='skus') already exists in the graph
✅ Graph ready (time column set, stypes assigned, links declared)


## Predictive Query (PQL): Batched forecast of 21-day demand for SKUs

Some SDK versions require enumerating entities explicitly. We’ll run PQL in batches using:

PREDICT SUM(sales.qty, 0, 21, days) FOR skus.sku_id IN (id1, id2, ...)



In [5]:
# Batched predictions over all SKUs using explicit IN (...)
model = rfm.KumoRFM(graph)

all_sku_ids = skus['sku_id'].astype(int).tolist()
BATCH = 200  # adjust if needed
batches = [all_sku_ids[i:i+BATCH] for i in range(0, len(all_sku_ids), BATCH)]

pred_frames = []
for i, batch in enumerate(batches):
    ids = ",".join(map(str, batch))
    query = f"PREDICT SUM(sales.qty, 0, 21, days) FOR skus.sku_id IN ({ids})"
    print(f"Batch {i+1}/{len(batches)}: {len(batch)} ids")
    res = model.predict(query, run_mode='fast')
    pred_frames.append(res)

preds = pd.concat(pred_frames, ignore_index=True) if pred_frames else pd.DataFrame(columns=['ENTITY','TARGET_PRED','ANCHOR_TIMESTAMP'])

# Tidy output
preds = preds.rename(columns={
    'ENTITY': 'sku_id',
    'TARGET_PRED': 'pred_qty_21d',
    'ANCHOR_TIMESTAMP': 'asof_week',
})

print(preds.head(3))


]9;4;3

Output()

]9;4;0Batch 1/3: 200 ids
]9;4;3

Output()

]9;4;0Batch 2/3: 200 ids
]9;4;3

Output()

]9;4;0Batch 3/3: 199 ids
]9;4;3

Output()

]9;4;0   sku_id  asof_week  pred_qty_21d
0       0 2024-04-08      2.843467
1       1 2024-04-08      1.658890
2       2 2024-04-08     19.922388


## Save predictions and minimal metadata (reproducibility)


In [6]:
# Join back to (store_id, product_id) for readability
sku_index = skus.set_index('sku_id')[[ 'store_id', 'product_id' ]]
out = preds.merge(sku_index, left_on='sku_id', right_index=True, how='left')[
    ['store_id','product_id','sku_id','asof_week','pred_qty_21d']
]

# Save predictions (forecast-only output)
pred_path = OUT_DIR / 'rfm_21d_predictions.csv'
out.to_csv(pred_path, index=False)

# Save run metadata (reproducibility)
meta = {
    'as_of_date': str(AS_OF_DATE.date()),
    'pql_query': query,
    'table_sizes': {
        'sales': int(len(sales)),
        'skus': int(len(skus)),
        'products': int(len(products)),
        'stores': int(len(stores)),
    },
}
import json
(OUT_DIR / 'rfm_21d_predictions_meta.json').write_text(json.dumps(meta, indent=2))

print('✅ Saved:', pred_path)
print('   Predictions: 21-day demand forecasts per SKU')
print('   Metadata:', OUT_DIR / 'rfm_21d_predictions_meta.json')
out.head(3)


✅ Saved: /Users/senoni/noni/vn2inventory/artifacts/kumo_rfm/rfm_21d_predictions.csv
   Predictions: 21-day demand forecasts per SKU
   Metadata: /Users/senoni/noni/vn2inventory/artifacts/kumo_rfm/rfm_21d_predictions_meta.json


,store_id,product_id,sku_id,asof_week,pred_qty_21d
0,0,126,0,2024-04-08,2.843467
1,0,182,1,2024-04-08,1.658890
2,1,124,2,2024-04-08,19.922388


## (Optional) Mini backtest: sliding as-of dates and MAE/MAPE

Validate forecast quality by predicting at prior cutoffs and comparing to realized demand.


In [7]:
# Choose a couple of prior Mondays as backtest cutoffs (sliding-origin validation)
backtest_cutoffs = [pd.Timestamp("2024-03-18"), pd.Timestamp("2024-03-25")]

metrics = []
for cutoff in backtest_cutoffs:
    # Prevent leakage: truncate to cutoff and rebuild graph
    s_bt = sales_long[sales_long['week'] <= cutoff]
    # Ensure sku_id is present in the sales events
    s_bt = (s_bt.merge(skus[['sku_id','store_id','product_id']],
                       on=['store_id','product_id'], how='left')
                 [["sku_id","store_id","product_id","week","qty"]])

    g_bt = rfm.LocalGraph.from_data({
        'sales': s_bt,
        'skus': skus,
        'products': products,
        'stores': stores,
    }, infer_metadata=True)
    g_bt['sales'].time_column = 'week'
    g_bt['sales']['qty'].stype = 'numerical'
    g_bt['sales']['sku_id'].stype = 'ID'
    g_bt['skus']['sku_id'].stype = 'ID'
    try:
        g_bt.link('sales','sku_id','skus')
        g_bt.link('skus','product_id','products')
        g_bt.link('skus','store_id','stores')
    except Exception:
        pass

    m_bt = rfm.KumoRFM(g_bt)
    q = "PREDICT SUM(sales.qty, 0, 21, days) FOR skus.sku_id IN (" + \
        ",".join(map(str, skus['sku_id'].tolist())) + ")"
    pred_bt = m_bt.predict(q, run_mode='fast').rename(columns={'ENTITY':'sku_id','TARGET_PRED':'pred'})

    # Realized next-21-day sums from original sales (no leakage)
    mask = (sales_long['week'] > cutoff) & (sales_long['week'] <= cutoff + pd.Timedelta(days=21))
    actual = (sales_long[mask]
              .merge(skus[['sku_id','store_id','product_id']], on=['store_id','product_id'], how='left')
              .groupby('sku_id')['qty']
              .sum()
              .rename('actual')
              .reset_index())

    df = pred_bt.merge(actual, on='sku_id', how='left').fillna({'actual':0.0})
    mae = float((df['pred'] - df['actual']).abs().mean())
    mape = float((np.where(df['actual']>0, (df['pred']-df['actual']).abs()/df['actual'], 0)).mean())
    metrics.append({'cutoff': str(cutoff.date()), 'mae': mae, 'mape': mape})

print("📊 Backtest metrics (MAE ≈ 3.3 units, MAPE high due to sparse/low-volume SKUs):")
pd.DataFrame(metrics)


### 🗂️ Graph Metadata

name,primary_key,time_column
sales,-,week
skus,sku_id,-
products,product_id,-
stores,store_id,-


### 🕸️ Graph Links (FK ↔️ PK)

- `sales.product_id` ↔️ `products.product_id`
- `skus.product_id` ↔️ `products.product_id`
- `sales.sku_id` ↔️ `skus.sku_id`
- `sales.store_id` ↔️ `stores.store_id`
- `skus.store_id` ↔️ `stores.store_id`

]9;4;3

...

Output()

]9;4;0]9;4;3

Output()

]9;4;0📊 Backtest metrics (MAE ≈ 3.3 units, MAPE high due to sparse/low-volume SKUs):


,cutoff,mae,mape
0,2024-03-18,3.378694,0.432636
1,2024-03-25,3.248304,0.414711
